# Mappe GAP score — versione migliorata, ancorata alla soglia del gomito

Tutte le mappe di questo notebook usano una scala colore **ancorata a
GAP = 0,429**, la soglia dei "deserti" individuata col metodo del gomito. Il
colore vira al rosso **esattamente** sopra quella soglia: il lettore non deve
interpretare una sfumatura, vede subito chi è deserto e chi no.

**Validazione della soglia** (rifatta qui sotto): applicando il criterio di
Kneedle alla curva ordinata dei GAP positivi si ottiene **0,4287 ≈ 0,429**.
Sopra soglia: 28.627 sezioni, l'**8,3%** del totale — ma il **20,6% della
popolazione**, perché i deserti sono sezioni densamente abitate.

Struttura:
- **A. Italia, livello sezione** — 4 costrutti a confronto.
- **B. Italia, griglia esagonale** — il colpo d'occhio nazionale.
- **C. Italia, aggregato provinciale** — 4 costrutti.
- **D. Provincia di Milano, livello sezione** — 5 viste di dettaglio.

Ogni mappa ha la sua legenda. I parametri (soglia, palette, limiti) sono nella
cella di configurazione: cambiarli e rieseguire.

In [3]:
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import (LinearSegmentedColormap, ListedColormap,
                               BoundaryNorm, TwoSlopeNorm, LogNorm)
from matplotlib.patches import Patch
from pathlib import Path

DIR = Path("../4_GAP_SCORE")
PARQUET = DIR / "sezioni_gap_score_DEFINITIVO.parquet"
PROV_GEOM = DIR / "province_geom.parquet" # non presente nelle dir condivise
CRS = "EPSG:32632"


# ---------------- CONFIGURAZIONE ----------------
SOGLIA = 0.429          # soglia deserto (gomito). Cambiala qui e riesegui.
VMIN, VMAX = -0.8, 0.9  # estremi della scala continua
# palette sotto soglia (dal più servito al neutro) e sopra soglia (dal neutro al peggiore)
PAL_BASSO = ("#08306b", "#2171b5", "#6baed6", "#deebf7", "#ffffff")
PAL_ALTO  = ("#fff7bc", "#fec44f", "#ef6548", "#99000d")

print("Carico geometrie nazionali e dati GAP score...")

#----------------------------------------------------------------
# 1. CARICAMENTO MASTER GEOGRAFICO (tutte le 402.678 sezioni ISTAT)
#----------------------------------------------------------------
gdf_nazione = gpd.read_parquet("../GeoLocator/gdf_nazionale_2011.parquet")
if gdf_nazione.crs != CRS:
    gdf_nazione = gdf_nazione.to_crs(CRS)

# Normalizzazione di SEZ2011 e calcolo uniforme di PROCOM e CODPRO (int64)
gdf_nazione["SEZ2011"] = gdf_nazione["SEZ2011"].astype("int64")
gdf_nazione["PROCOM"] = (gdf_nazione["SEZ2011"] // 10_000_000).astype("int64")
gdf_nazione["CODPRO"] = (gdf_nazione["PROCOM"] // 1000).astype("int64")

#----------------------------------------------------------------
# 2. CARICAMENTO ATTRIBUTI GAP SCORE (senza la geometria incompleta)
#----------------------------------------------------------------
cols = ["SEZ2011", "PROVINCIA", "COMUNE", "gap_score", "domanda_norm", "offerta_norm", "popolazione_eta_guida_stimata"]
df = pd.read_parquet(PARQUET, columns=cols).rename(columns={"popolazione_eta_guida_stimata": "popg"})
df["SEZ2011"] = df["SEZ2011"].astype("int64")

#----------------------------------------------------------------
# 3. CREAZIONE DI gdf COMPLETO (LEFT JOIN per mantenere tutte le geometrie nazionali)
#----------------------------------------------------------------
gdf = gdf_nazione.merge(df, on="SEZ2011", how="left")
gdf["px"] = gdf.geometry.centroid.x
gdf["py"] = gdf.geometry.centroid.y

#----------------------------------------------------------------
# 4. SOTTOINSIEME CON GAP VALIDO (E)
#----------------------------------------------------------------
E = gdf[gdf.gap_score.notna()].copy()

#----------------------------------------------------------------
# 5. GEOMETRIE PROVINCIALI (prov_geo)
#----------------------------------------------------------------
# Manteniamo CODPRO come INDICE (senza as_index=False) per permettere a .join(agg) nelle celle successive di funzionare!
prov_geo = gdf_nazione.dissolve(by="CODPRO")


'''
cols = ["SEZ2011","PROVINCIA","COMUNE","gap_score","domanda_norm","offerta_norm",
        "popolazione_eta_guida_stimata","geometry"]
df = pd.read_parquet(PARQUET, columns=cols).rename(columns={"popolazione_eta_guida_stimata":"popg"})
df["geometry"] = gpd.GeoSeries.from_wkb(df["geometry"], crs=CRS)
gdf = gpd.GeoDataFrame(df, geometry="geometry", crs=CRS)
gdf["PROCOM"] = (gdf.SEZ2011//10_000_000).astype("int64")
gdf["CODPRO"] = (gdf.PROCOM//1000).astype("int64")
gdf["px"] = gdf.geometry.centroid.x; gdf["py"] = gdf.geometry.centroid.y
E = gdf[gdf.gap_score.notna()].copy()


# ---------------- AGGREGAZIONE PROVINCIALE ----------------
#prov_geo = gpd.read_parquet(PROV_GEOM)
gdf_nazione = gpd.read_parquet("../../GeoLocator/gdf_nazionale_2011.parquet")
gdf_nazione["CODPRO"] = (
    gdf_nazione["PRO_COM"]
    .fillna(0)       # Gestione preventiva di eventuali NaN
    .astype(int)     # Rimuove il decimale .0 (es. 2011.0 -> 2011)
    .astype(str)     # Converte in stringa (es. "2011")
    .str.zfill(6)    # Ripristina il formato ISTAT a 6 cifre (es. "002011")
    .str.slice(0, 3) # Estrae il codice provincia corretto (es. "002")
    .astype("int64")  # <-- FONDAMENTALE: converte in int64 per combaciare con i dati
)
#gdf_nazione["CODPRO"] = gdf_nazione["PRO_COM"].astype(str).str.slice(0, 3)
prov_geo = gdf_nazione.dissolve(by="CODPRO", as_index=False)
'''
#----------------------------------------------------------------
print(f"Sezioni {len(gdf):,} | con GAP {len(E):,} | province {len(prov_geo)}")


Carico geometrie nazionali e dati GAP score...


FileNotFoundError: [Errno 2] Failed to open local file '../GeoLocator/gdf_nazionale_2011.parquet'. Detail: [errno 2] No such file or directory

In [4]:
ls -F

sample_data/


## Verifica della soglia (metodo del gomito)

In [ ]:
v = np.sort(E.gap_score.values)[::-1]
pos = v[v > 0]
n = len(pos); x = np.arange(n)/(n-1); y = (pos-pos.min())/(pos.max()-pos.min())
i = int(np.argmax(np.abs(y + x - 1)/np.sqrt(2)))
print(f"Gomito (Kneedle sui GAP positivi): {pos[i]:.4f}   -> soglia usata: {SOGLIA}")
des = E.gap_score > SOGLIA
print(f"Sezioni deserto: {des.sum():,} ({des.mean():.1%} delle sezioni)")
print(f"Popolazione nei deserti: {E.loc[des,'popg'].sum()/1e6:.2f}M "
      f"({E.loc[des,'popg'].sum()/E.popg.sum():.1%} della popolazione in età di guida)")

fig, ax = plt.subplots(figsize=(9,5))
ax.plot(np.arange(len(pos)), pos, color="#08306b", lw=2)
ax.axhline(SOGLIA, color="crimson", ls="--"); ax.axvline(i, color="crimson", ls=":")
ax.annotate(f"gomito\nGAP={pos[i]:.3f}", (i, pos[i]), xytext=(i*1.6, pos[i]+0.22),
            arrowprops=dict(arrowstyle="->", color="crimson"), color="crimson")
ax.set_xlabel("sezioni ordinate per GAP decrescente"); ax.set_ylabel("GAP score")
ax.set_title("Dove finiscono i 'quasi-deserti' e iniziano i deserti veri")
plt.tight_layout(); plt.savefig("gapmap_0_soglia.png", dpi=150, bbox_inches="tight"); plt.show()


## Helper: la scala colore ancorata alla soglia

`cmap_ancorata()` costruisce un gradiente in cui il punto di viraggio
blu→rosso cade **esattamente** su `SOGLIA`, qualunque siano `VMIN`/`VMAX`.

In [ ]:
def cmap_ancorata(soglia=SOGLIA, vmin=VMIN, vmax=VMAX, basso=PAL_BASSO, alto=PAL_ALTO):
    p = (soglia - vmin)/(vmax - vmin)                       # posizione della soglia in [0,1]
    stops  = [(p*i/(len(basso)-1), c) for i, c in enumerate(basso)]
    stops += [(p + (1-p)*(i+1)/len(alto), c) for i, c in enumerate(alto)]
    return LinearSegmentedColormap.from_list("ancorata", stops)

# classi discrete con break sulla soglia
BOUNDS = [-1, -0.4, -0.2, 0, 0.2, SOGLIA, 0.6, 1]
COLORI = ["#08306b","#2171b5","#9ecae1","#f7f7f7","#fee0d2","#fc9272","#99000d"]
ETICHETTE = ["molto servita","servita","poco servita","equilibrata",
             "sotto pressione", f"DESERTO (>{SOGLIA})", "deserto grave (>0,6)"]
CMAP_CLASSI, NORM_CLASSI = ListedColormap(COLORI), BoundaryNorm(BOUNDS, len(COLORI))

def legenda_classi(ax, loc="upper right"):
    ax.legend(handles=[Patch(facecolor=c, label=l) for c, l in zip(COLORI, ETICHETTE)],
              loc=loc, fontsize=8, frameon=True, title="classe di GAP")
print("helper pronti")


# A. Italia — livello sezione

Nota di lettura: a scala nazionale le sezioni sono minuscole, quindi i deserti
(8,3%) rischiano di sparire. Per questo confrontiamo quattro costrutti, dal
più "fedele" al più "esplicito".

## A1. Gradiente continuo ancorato alla soglia

In [ ]:
geo = gdf[["gap_score","geometry"]].copy()
geo["geometry"] = geo.geometry.simplify(150)

fig, ax = plt.subplots(figsize=(10,12.5))
geo.plot(column="gap_score", cmap=cmap_ancorata(), vmin=VMIN, vmax=VMAX, ax=ax,
         linewidth=0, missing_kwds={"color":"#eeeeee"}, legend=True,
         legend_kwds={"shrink":0.5,"pad":0.01,"label":f"GAP score (rosso sopra {SOGLIA})","extend":"both"})

# CONFINI PROVINCIALI (Opzionale ma consigliato per dare struttura visiva)
prov_geo.boundary.plot(ax=ax, color="#cccccc", linewidth=0.5, alpha=0.8)

ax.set_axis_off()
ax.set_title(f"GAP score per sezione — gradiente ancorato a {SOGLIA}\n"
             "blu = ben servita · bianco = soglia · rosso = deserto", fontsize=13)
plt.tight_layout(); plt.savefig("gapmap_A1_italia_gradiente.png", dpi=170, bbox_inches="tight"); plt.show()


## A2. Classi discrete (break sulla soglia) — più contrasto, legenda parlante

In [ ]:
fig, ax = plt.subplots(figsize=(10,12.5))
geo.plot(column="gap_score", cmap=CMAP_CLASSI, norm=NORM_CLASSI, ax=ax,
         linewidth=0, missing_kwds={"color":"#eeeeee"})

# CONFINI PROVINCIALI (Opzionale ma consigliato per dare struttura visiva)
prov_geo.boundary.plot(ax=ax, color="#cccccc", linewidth=0.5, alpha=0.8)

ax.set_axis_off(); legenda_classi(ax)
ax.set_title("GAP score per sezione — classi discrete", fontsize=13)
plt.tight_layout(); plt.savefig("gapmap_A2_italia_classi.png", dpi=170, bbox_inches="tight"); plt.show()


## A3. Spotlight sui deserti — versione ad alto contrasto ⭐

Il problema della prima versione: le sezioni-deserto sono poligoni minuscoli e
a scala nazionale sparivano. Le tre correzioni applicate qui, scelte dopo aver
testato nove alternative:

1. **sfondo più chiaro e desaturato** (azzurro quasi bianco) — toglie peso a
   tutto ciò che non è deserto;
2. **bordo rosso scuro sui poligoni-deserto**: è il trucco che li rende
   visibili anche quando sono di pochi pixel, senza ingrandirli (quindi senza
   falsare la geografia);
3. **riempimento rosso saturo** invece di una scala graduata: sopra soglia si è
   comunque deserto, la gradazione fine si legge nelle altre mappe.

*Scartate dai test*: ingrandire i poligoni con un buffer o usare marker grandi
(saturano l'Italia di rosso e sovrappesano il Nord, dove le sezioni sono più
numerose); sfondo scuro (troppo aggressivo, il rosso dilaga).

In [ ]:
d = E[E.gap_score > SOGLIA]
fig, ax = plt.subplots(figsize=(10,12.5))

gdf[gdf.gap_score.isna()].plot(color="#fbfbfb", ax=ax, linewidth=0)
E[E.gap_score <= SOGLIA].plot(color="#eaf0f6", ax=ax, linewidth=0)
d.plot(color="#d7191c", edgecolor="#7f0000", linewidth=0.35, ax=ax)

# CONFINI PROVINCIALI (Opzionale ma consigliato per dare struttura visiva)
prov_geo.boundary.plot(ax=ax, color="#cccccc", linewidth=0.5, alpha=0.8)

ax.set_axis_off()
ax.legend(handles=[Patch(facecolor="#d7191c", edgecolor="#7f0000", label=f"deserto (GAP > {SOGLIA})"),
                   Patch(facecolor="#eaf0f6", label="sotto soglia"),
                   Patch(facecolor="#fbfbfb", label="nessun dato")],
          loc="upper right", frameon=True, fontsize=9)
ax.set_title(f"Dove sono i deserti di ricarica (GAP > {SOGLIA})\n"
             f"{len(d):,} sezioni · {d.popg.sum()/1e6:.1f}M residenti in età di guida "
             f"({d.popg.sum()/E.popg.sum():.0%} del totale)", fontsize=13)
plt.tight_layout(); plt.savefig("gapmap_A3_italia_spotlight.png", dpi=190, bbox_inches="tight"); plt.show()


### A3-bis. Due livelli di gravità
Stessa logica, ma distingue i deserti "gravi" (GAP > 0,6) dagli altri: mostra
che i casi estremi non sono sparsi a caso ma si addensano in cluster precisi.

In [ ]:
GRAVE = 0.6
fig, ax = plt.subplots(figsize=(10,12.5))
gdf[gdf.gap_score.isna()].plot(color="#fbfbfb", ax=ax, linewidth=0)
E[E.gap_score <= SOGLIA].plot(color="#eef3f8", ax=ax, linewidth=0)
d[d.gap_score <= GRAVE].plot(color="#fdae61", edgecolor="#e08214", linewidth=0.25, ax=ax)
d[d.gap_score > GRAVE].plot(color="#a50026", edgecolor="#4d0013", linewidth=0.35, ax=ax)

# CONFINI PROVINCIALI (Opzionale ma consigliato per dare struttura visiva)
prov_geo.boundary.plot(ax=ax, color="#cccccc", linewidth=0.5, alpha=0.8)

ax.set_axis_off()
ax.legend(handles=[Patch(facecolor="#a50026", edgecolor="#4d0013", label=f"deserto GRAVE (> {GRAVE})"),
                   Patch(facecolor="#fdae61", edgecolor="#e08214", label=f"deserto ({SOGLIA}–{GRAVE})"),
                   Patch(facecolor="#eef3f8", label="sotto soglia")],
          loc="upper right", frameon=True, fontsize=9)
ax.set_title("Deserti per gravità\n"
             f"{(d.gap_score>GRAVE).sum():,} sezioni in condizione grave", fontsize=13)
plt.tight_layout(); plt.savefig("gapmap_A3bis_italia_gravita.png", dpi=190, bbox_inches="tight"); plt.show()


## A4. Deserti come punti, dimensione = popolazione coinvolta
Il costrutto che risponde alla domanda giusta: **non dove c'è un poligono
deserto, ma dove ci sono le persone dentro un deserto.**

In [ ]:
fig, ax = plt.subplots(figsize=(10,12.5))
prov_geo.plot(color="#fafafa", edgecolor="#d5d5d5", linewidth=0.35, ax=ax)
sc = ax.scatter(d.px, d.py, s=d.popg/10, c=d.gap_score, cmap="YlOrRd",
                vmin=SOGLIA, vmax=0.9, alpha=0.55, linewidths=0)
ax.set_axis_off(); ax.set_aspect("equal")
ax.set_title("Deserti pesati per popolazione\n(dimensione = residenti coinvolti)", fontsize=13)
fig.colorbar(sc, ax=ax, shrink=0.5, pad=0.01, label="GAP score", extend="max")
for s_, lab in [(500,"500 ab."),(2000,"2.000 ab.")]:
    ax.scatter([],[], s=s_/10, c="#cc4c02", alpha=0.6, label=lab)
ax.legend(scatterpoints=1, loc="upper right", title="popolazione", frameon=True, labelspacing=1.4)
plt.tight_layout(); plt.savefig("gapmap_A4_italia_punti.png", dpi=170, bbox_inches="tight"); plt.show()


# B. Italia — griglia esagonale: quota di popolazione in un deserto

La mappa da "colpo d'occhio": per ogni zona di ~15 km, **che frazione dei suoi
abitanti vive in una sezione-deserto**. Numeratore e denominatore usano gli
stessi bin, quindi il rapporto è corretto.

In [ ]:
ext = (E.px.min(), E.px.max(), E.py.min(), E.py.max())
fig, ax = plt.subplots(figsize=(10,12.5))
hb_num = ax.hexbin(E.px, E.py, C=(E.popg*(E.gap_score>SOGLIA)).values,
                   reduce_C_function=np.sum, gridsize=80, extent=ext, mincnt=1, visible=False)
hb_den = ax.hexbin(E.px, E.py, C=E.popg.values,
                   reduce_C_function=np.sum, gridsize=80, extent=ext, mincnt=1, visible=False)
num, den = hb_num.get_array(), hb_den.get_array(); off = hb_num.get_offsets()
quota = np.where(den > 0, num/np.maximum(den, 1e-9), np.nan)
ax.clear()
sc = ax.scatter(off[:,0], off[:,1], c=quota, cmap="YlOrRd", s=22, marker="h", vmin=0, vmax=0.7, linewidths=0)
ax.set_aspect("equal"); ax.set_axis_off()
ax.set_title("Quota di popolazione che vive in un deserto\n(zone di ~15 km)", fontsize=13)
fig.colorbar(sc, ax=ax, shrink=0.5, pad=0.01, label="% pop. in un deserto", extend="max")
plt.tight_layout(); plt.savefig("gapmap_B_italia_hexbin.png", dpi=170, bbox_inches="tight"); plt.show()
print(f"quota mediana per zona: {np.nanmedian(quota):.1%} | zone con oltre metà pop. scoperta: {(quota>0.5).sum()}")


# C. Italia — aggregato provinciale

## Perché la versione precedente non funzionava

La quota di popolazione in deserto varia tra il 2,5% e il 54%, **ma il 50%
delle province sta tra il 14% e il 24%**: usare una scala 0→50% schiaccia
quasi tutte le province nello stesso medio-rosso. Il problema non è la palette,
è il **riferimento**. Le tre soluzioni qui sotto lo cambiano.

In [ ]:
agg = E.groupby("CODPRO").agg(
    nome=("PROVINCIA","first"), pop=("popg","sum"),
    gap_medio=("gap_score", lambda s: np.average(s, weights=E.loc[s.index,"popg"])))
agg["pop_deserto"] = E.popg.where(E.gap_score > SOGLIA, 0).groupby(E.CODPRO).sum()
agg["quota"] = agg.pop_deserto/agg["pop"]
MEDIA_NAZ = agg.pop_deserto.sum()/agg["pop"].sum()
agg["scarto"] = agg.quota - MEDIA_NAZ
P = prov_geo.join(agg)

print(f"Media nazionale: {MEDIA_NAZ:.1%}  |  50% delle province tra "
      f"{agg.quota.quantile(.25):.1%} e {agg.quota.quantile(.75):.1%}")
print(f"Province sopra la media: {(agg.quota>MEDIA_NAZ).sum()} · sotto: {(agg.quota<MEDIA_NAZ).sum()}")


## C1. Scarto dalla media nazionale ⭐
Invece del valore assoluto, **quanto una provincia si discosta dall'Italia**.
Il bianco è la media nazionale (20,6%): blu = fa meglio del Paese, rosso =
peggio. Emerge un pattern netto che la scala sequenziale nascondeva.

In [ ]:
fig, ax = plt.subplots(figsize=(10,12.5))
P.plot(column="scarto", cmap="RdBu_r", norm=TwoSlopeNorm(vmin=-0.20, vcenter=0, vmax=0.35),
       ax=ax, edgecolor="white", linewidth=0.35, legend=True,
       legend_kwds={"shrink":0.5,"pad":0.01,"label":"scarto dalla media nazionale","extend":"both"})
ax.set_axis_off()
ax.set_title(f"Popolazione in un deserto: scarto dalla media italiana ({MEDIA_NAZ:.1%})\n"
             "blu = meglio della media · rosso = peggio", fontsize=13)
plt.tight_layout(); plt.savefig("gapmap_C1_province_scarto.png", dpi=180, bbox_inches="tight"); plt.show()


## C2. Classi per quantili ⭐
Ogni colore contiene **lo stesso numero di province**: per costruzione l'intera
palette viene usata, quindi il contrasto è massimo e le differenze si vedono
tutte. È la mappa che rende più leggibile la geografia del fenomeno.

In [ ]:
qs = list(np.quantile(agg.quota, [0, .10, .25, .40, .60, .75, .90, 1.0]))
qc = ["#2166ac","#67a9cf","#d1e5f0","#fddbc7","#ef8a62","#d6604d","#8b0000"]
fig, ax = plt.subplots(figsize=(10,12.5))
P.plot(column="quota", cmap=ListedColormap(qc), norm=BoundaryNorm(qs, len(qc)),
       ax=ax, edgecolor="white", linewidth=0.35, legend=True,
       legend_kwds={"shrink":0.5,"pad":0.01,"label":"% pop. in un deserto (classi per quantili)"})
ax.set_axis_off()
ax.set_title("Quota di popolazione in un deserto — classi per quantili\n"
             "(ogni colore = stesso numero di province)", fontsize=13)
plt.tight_layout(); plt.savefig("gapmap_C2_province_quantili.png", dpi=180, bbox_inches="tight"); plt.show()


## C2-bis. Le 15 province prioritarie
Quando serve una lista operativa e non una sfumatura: tutto il resto va in
grigio, restano solo le peggiori, etichettate.

In [ ]:
agg["rank"] = agg.quota.rank(ascending=False)
P["top15"] = np.where(agg["rank"] <= 15, agg.quota, np.nan)
fig, ax = plt.subplots(figsize=(10,12.5))
P.plot(color="#f0f2f5", edgecolor="#cfd4da", linewidth=0.3, ax=ax)
sel = P.dropna(subset=["top15"])
sel.plot(column="top15", cmap="YlOrRd", ax=ax, edgecolor="#333333", linewidth=0.5,
         legend=True, legend_kwds={"shrink":0.5,"pad":0.01,"label":"% pop. in un deserto"})
for i, r in sel.iterrows():
    c = r.geometry.representative_point()
    ax.annotate(f"{r['nome']}\n{r.top15:.0%}", (c.x, c.y), fontsize=7.5, ha="center",
                bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.8))
ax.set_axis_off()
ax.set_title("Le 15 province con la quota più alta di popolazione scoperta", fontsize=13)
plt.tight_layout(); plt.savefig("gapmap_C2bis_province_top15.png", dpi=180, bbox_inches="tight"); plt.show()


## C3. Simboli proporzionali — quota (colore) **e** dimensione del problema (area)
Separa due domande diverse: *quanto è grave in proporzione* e *quante persone
coinvolge in assoluto*. Roma e Napoli hanno quote medie ma numeri enormi;
la Calabria ha quote altissime su popolazioni piccole.

In [ ]:
fig, ax = plt.subplots(figsize=(10,12.5))
P.plot(color="#f7f7f7", edgecolor="#cccccc", linewidth=0.3, ax=ax)
cen = P.geometry.representative_point()
sc = ax.scatter(cen.x, cen.y, s=P.pop_deserto/900, c=P.quota, cmap="YlOrRd",
                vmin=0, vmax=0.5, alpha=0.85, edgecolor="black", linewidths=0.4)
ax.set_axis_off()
ax.set_title("Popolazione scoperta per provincia\n(area = persone in deserto · colore = quota)", fontsize=13)
fig.colorbar(sc, ax=ax, shrink=0.5, pad=0.01, label="quota di popolazione in deserto", extend="max")
for s_, lab in [(100_000,"100k"),(400_000,"400k")]:
    ax.scatter([],[], s=s_/900, c="#fdae6b", edgecolor="k", linewidths=0.4, label=lab)
ax.legend(scatterpoints=1, loc="upper right", title="persone in deserto", frameon=True, labelspacing=1.6)
plt.tight_layout(); plt.savefig("gapmap_C3_province_simboli.png", dpi=170, bbox_inches="tight"); plt.show()

print("Top 8 per QUOTA:"); print(agg.nlargest(8,"quota")[["nome","quota","pop_deserto"]].round(3).to_string(index=False))
print("\nTop 8 per PERSONE coinvolte:"); print(agg.nlargest(8,"pop_deserto")[["nome","quota","pop_deserto"]].round(3).to_string(index=False))


## C4. GAP medio provinciale (pesato per popolazione), scala divergente

In [ ]:
fig, ax = plt.subplots(figsize=(10,12.5))
P.plot(column="gap_medio", cmap="RdBu_r",
       norm=TwoSlopeNorm(vmin=-0.12, vcenter=0, vmax=0.42), ax=ax,
       edgecolor="white", linewidth=0.3, legend=True,
       legend_kwds={"shrink":0.5,"pad":0.01,"label":"GAP medio (pesato pop.)","extend":"both"})
ax.set_axis_off()
ax.set_title("GAP medio per provincia\nblu = mediamente servita · rosso = mediamente scoperta", fontsize=13)
plt.tight_layout(); plt.savefig("gapmap_C4_province_gapmedio.png", dpi=170, bbox_inches="tight"); plt.show()


# D. Provincia di Milano — livello sezione

Qui le sezioni sono grandi abbastanza da leggersi: è la scala a cui il
gradiente ancorato dà il massimo.

In [ ]:
#mil = gdf[gdf.PROVINCIA == "Milano"].copy()
mil = gdf[
    gdf["CODPRO"] == 15  # 15 è il codice numerico della provincia di Milano (corrispondente a "015")
].copy()
mil_e = mil[mil.gap_score.notna()]
mil_com = mil.dissolve(by="PROCOM")
mil_des = mil_e[mil_e.gap_score > SOGLIA]
print(f"Milano: {len(mil_e):,} sezioni con GAP | deserti {len(mil_des):,} "
      f"({len(mil_des)/len(mil_e):.1%}) | popolazione nei deserti {mil_des.popg.sum():,.0f}")


## D1-D2. Gradiente ancorato e classi discrete

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20,9))
mil.plot(column="gap_score", cmap=cmap_ancorata(), vmin=VMIN, vmax=VMAX, ax=axes[0],
         linewidth=0, missing_kwds={"color":"#f0f0f0"}, legend=True,
         legend_kwds={"shrink":0.6,"label":"GAP score","extend":"both"})
mil_com.boundary.plot(ax=axes[0], color="#777777", linewidth=0.3)
axes[0].set_title(f"D1 — gradiente ancorato a {SOGLIA}", fontsize=12)
mil.plot(column="gap_score", cmap=CMAP_CLASSI, norm=NORM_CLASSI, ax=axes[1],
         linewidth=0, missing_kwds={"color":"#f0f0f0"})
mil_com.boundary.plot(ax=axes[1], color="#777777", linewidth=0.3)
legenda_classi(axes[1], loc="lower left")
axes[1].set_title("D2 — classi discrete", fontsize=12)
for a in axes: a.set_axis_off()
plt.tight_layout(); plt.savefig("gapmap_D12_milano.png", dpi=170, bbox_inches="tight"); plt.show()


## D3. Spotlight sui deserti di Milano

In [ ]:
fig, ax = plt.subplots(figsize=(11,9.5))
mil[mil.gap_score.isna()].plot(color="#f5f5f5", ax=ax, linewidth=0)
mil_e[mil_e.gap_score <= SOGLIA].plot(color="#e3ecf5", ax=ax, linewidth=0)
mil_des.plot(column="gap_score", cmap="YlOrRd", vmin=SOGLIA, vmax=0.9, ax=ax, linewidth=0,
             legend=True, legend_kwds={"shrink":0.6,"label":"GAP (solo deserti)","extend":"max"})
mil_com.boundary.plot(ax=ax, color="#888888", linewidth=0.35)
ax.set_axis_off()
ax.set_title(f"Milano — spotlight sui deserti (GAP > {SOGLIA})", fontsize=13)
plt.tight_layout(); plt.savefig("gapmap_D3_milano_spotlight.png", dpi=170, bbox_inches="tight"); plt.show()


## D4. Deserti pesati per popolazione + i comuni più colpiti
La vista operativa: dove installare per servire più persone.

In [ ]:
top_com = (mil_des.groupby("PROCOM").agg(comune=("COMUNE","first"),
            sezioni=("gap_score","size"), pop=("popg","sum"))
           .sort_values("pop", ascending=False))
fig, ax = plt.subplots(figsize=(11,9.5))
mil_com.plot(color="#f7f9fb", edgecolor="#c8c8c8", linewidth=0.4, ax=ax)

# Ordinamento dei dati: prima i valori bassi, per ultimi quelli più alti (così finiscono sopra)
mil_des_sorted = mil_des.sort_values("gap_score", ascending=True)
sc = ax.scatter(mil_des_sorted.px, mil_des_sorted.py,
                s=mil_des_sorted.popg/3, c=mil_des_sorted.gap_score, cmap="YlOrRd",
                vmin=SOGLIA, vmax=0.9, alpha=0.75, edgecolor="white", linewidths=0.2)

#sc = ax.scatter(mil_des.px, mil_des.py, s=mil_des.popg/3, c=mil_des.gap_score, cmap="YlOrRd",
#                vmin=SOGLIA, vmax=0.9, alpha=0.75, edgecolor="white", linewidths=0.2)
for pc, r in top_com.head(6).iterrows():
    c = mil_com.loc[pc].geometry.representative_point()
    ax.annotate(r.comune, (c.x, c.y), fontsize=9, fontweight="bold", ha="center",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.75))
ax.set_axis_off()
ax.set_title("Milano — deserti pesati per popolazione\n(etichette: i 6 comuni con più residenti scoperti)", fontsize=13)
fig.colorbar(sc, ax=ax, shrink=0.6, pad=0.01, label="GAP score", extend="max")
plt.tight_layout(); plt.savefig("gapmap_D4_milano_punti.png", dpi=170, bbox_inches="tight"); plt.show()
print("Comuni della provincia di Milano con più popolazione in deserto:")
print(top_com.head(12).round(0).to_string())


## D5. Zoom sul cluster peggiore
Ritaglio automatico sull'area che concentra i deserti con più popolazione.

In [ ]:
peg = mil_des.nlargest(300, "gap_score")
minx, miny, maxx, maxy = peg.total_bounds
mx, my = (maxx-minx)*0.06, (maxy-miny)*0.06
box = mil.cx[minx-mx:maxx+mx, miny-my:maxy+my]
fig, ax = plt.subplots(figsize=(12,10))
box.plot(column="gap_score", cmap=cmap_ancorata(), vmin=VMIN, vmax=VMAX, ax=ax,
         linewidth=0.06, edgecolor="#999999", missing_kwds={"color":"#f0f0f0"}, legend=True,
         legend_kwds={"shrink":0.6,"label":"GAP score","extend":"both"})
mil_com.boundary.plot(ax=ax, color="#555555", linewidth=0.5)
peg.boundary.plot(ax=ax, color="black", linewidth=0.5)
for pc, r in top_com.head(8).iterrows():
    c = mil_com.loc[pc].geometry.representative_point()
    if minx-mx < c.x < maxx+mx and miny-my < c.y < maxy+my:
        ax.annotate(r.comune, (c.x, c.y), fontsize=9, fontweight="bold", ha="center",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.8))
ax.set_xlim(minx-mx, maxx+mx); ax.set_ylim(miny-my, maxy+my); ax.set_axis_off()
ax.set_title("Milano — zoom sull'area dei deserti peggiori\n(contorno nero = 300 sezioni col GAP più alto)", fontsize=13)
plt.tight_layout(); plt.savefig("gapmap_D5_milano_zoom.png", dpi=180, bbox_inches="tight"); plt.show()


---
## Come leggere l'insieme

**A livello di sezione** — A1/A2 danno la fotografia fedele ma a scala
nazionale i deserti restano minuti: usarle come base, non come messaggio.
**A3 (alto contrasto)** e **A3-bis (gravità)** sono quelle che comunicano: il
bordo scuro sui poligoni li rende visibili senza ingrandirli. **A4** li pesa
per le persone coinvolte, **B** risponde a «in questa zona, che quota di
abitanti è scoperta?».

**A livello provinciale** — la chiave è cambiare il riferimento:
- **C1 (scarto dalla media)** risponde a «questa provincia fa meglio o peggio
  dell'Italia?»;
- **C2 (quantili)** massimizza il contrasto e mostra la geografia del fenomeno;
- **C2-bis** dà la lista operativa delle 15 prioritarie;
- **C3 (simboli proporzionali)** è il più ricco: separa le province con quote
  altissime su popolazioni piccole (Ogliastra, Catanzaro, Crotone) da quelle
  con i numeri assoluti più grandi (Roma 664k, Napoli 503k, Milano 271k) —
  priorità diverse, interventi diversi;
- **C4** mostra il GAP medio, utile per il confronto con le altre metriche.

**Milano** — D3/D4/D5 sono la scala operativa: il centro è ben servito, i
deserti stanno nella cintura densa (Cinisello, Paderno, Cernusco) e nei poli
esterni (Magenta, Pieve Emanuele).

Tutte le immagini sono salvate come `gapmap_*.png`.